In [16]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import json

# Carrega uma única vez as variáveis do arquivo .env
load_dotenv(override=True)

True

## Teste com um arquivo

In [24]:
from pathlib import Path

arquivo_md = Path("Arquivos/md/bioetica_e_ia.md")

texto = arquivo_md.read_text(
    encoding="utf-8",
    errors="ignore"
)
print(type(texto))
print(len(texto))
print(texto[:500])   # primeiros 500 caracteres

<class 'str'>
51275
---
noteId: "1e073890911011f1994dc5ca4f139fa1"
tags: []

---

273

<!-- image -->

## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial

Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1

1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.

## Resumo

O avanço da inteligência artificial tem transformado profundamente a prática médica. De sistemas de apoio à decisão clínica a algoritmos de triagem e diagnóstico, a inteligência a


<class 'str'>
51275
---
noteId: "1e073890911011f1994dc5ca4f139fa1"
tags: []

---

273

<!-- image -->

## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial

Juracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1

1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.

## Resumo

O avanço da inteligência artificial tem transformado profundamente a prática médica. De sistemas de apoio à decisão clínica a algoritmos de triagem e diagnóstico, a inteligência a


## Definir o que o agente deve fazer.

In [4]:
messages = {
    "role": "system",
    "content": (
        "Extraia informações de artigos científicos."
    )
},
{
    "role": "user",
    "content": (
        "Extraia o título, autores e ano."
    )
},
{
    "role": "user",
    "content": texto
}

{'role': 'user',
 'content': '---\nnoteId: "1e073890911011f1994dc5ca4f139fa1"\ntags: []\n\n---\n\n273\n\n<!-- image -->\n\n## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial\n\nJuracy Barbosa dos Santos 1 , Guilhermina Rego 1 , Rui Nunes 1\n\n1. Faculdade de Medicina da Universidade do Porto, Porto, Portugal.\n\n## Resumo\n\nO avanço da inteligência artificial tem transformado profundamente a prática médica. De sistemas de apoio à decisão clínica a algoritmos de triagem e diagnóstico, a inteligência artificial tem demons -trado potencial para diagnósticos precoces, terapias personalizadas, otimização de recursos, redução de erros e ampliação do acesso a cuidados especializados. Contudo, essa revolução tecnológica impõe desafios éticos significativos aos princípios clássicos da bioética, como beneficência, não maleficência, confidencialidade e respeito à autonomia do doente, consagrados desde o Juramento de Hipócrates. A incorporação da inteligê

## Alguns modelos não suportam a definição do formato de resposta.

In [ ]:
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError(
        "A variável OPENROUTER_API_KEY não foi encontrada no .env"
    )

print("Conectando ao OpenRouter...")

resposta_openrouter = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {openrouter_api_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "openai/gpt-4o-mini", # Este modelo nao suporta response_format ->"openai/gpt-3.5-turbo",
        "messages": [
             {
    "role": "system",
    "content": (
        "Extraia informações de artigos científicos."
    )
},            

            {
                "role": "user",
                "content": texto,
                
            }
        ],
        'response_format': {
            "type": "json_schema",
            "json_schema": {
                "name": "article_info",
                "schema": {
                    "type": "object",
                    "properties": {
                        "titulo": {"type": "string"},
                        "autores": {"type": "string"},
                        "ano_publicacao": {"type": "string"}
                    },
                    "required": ["titulo", "autores", "ano_publicacao"]
                }
            }
        }
    },
    timeout=60
)

# Interrompe em erros HTTP, como 401, 404 ou 429
try:
    resposta_openrouter.raise_for_status()
except requests.exceptions.HTTPError:
    print("\nErro HTTP no OpenRouter:")
    print("Status:", resposta_openrouter.status_code)
    print("Resposta:", resposta_openrouter.text)
else:
    resultado_openrouter = resposta_openrouter.json()

    if "choices" in resultado_openrouter:
        print("\nResposta do OpenRouter")
        #print(
        #    resultado_openrouter["choices"][0]
        #    ["message"]["content"]
        #)
        data = json.loads(
            resultado_openrouter["choices"][0]["message"]["content"]
        )
        print("Dataframe com os dados extraídos do artigo:")
        display(pd.DataFrame([data]))
        
    elif "error" in resultado_openrouter:
        print("\nErro retornado pelo OpenRouter:")
        print(resultado_openrouter["error"])

    else:
        print("\nResposta inesperada do OpenRouter:")
        print(resultado_openrouter)

Conectando ao OpenRouter...

Resposta do OpenRouter
Dataframe com os dados extraídos do artigo:


,titulo,autores,ano_publicacao
0,Entre o algoritmo e o Juramento de Hipócrates:...,"Juracy Barbosa dos Santos, Guilhermina Rego, R...",2026


In [23]:
print(json.dumps(resultado_openrouter, indent=4, ensure_ascii=False))

{
    "id": "gen-1786020071-OsO5Vtq7qGoLN9ETlDuw",
    "object": "chat.completion",
    "created": 1786020071,
    "model": "openai/gpt-4o-mini",
    "provider": "Azure",
    "system_fingerprint": "fp_27599ce29d",
    "service_tier": null,
    "choices": [
        {
            "index": 0,
            "logprobs": null,
            "finish_reason": "stop",
            "native_finish_reason": "stop",
            "message": {
                "role": "assistant",
                "content": "{\"titulo\":\"Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial\",\"autores\":\"Juracy Barbosa dos Santos, Guilhermina Rego, Rui Nunes\",\"ano_publicacao\":\"2026\"}",
                "refusal": null,
                "reasoning": null
            }
        }
    ],
    "usage": {
        "prompt_tokens": 12590,
        "completion_tokens": 50,
        "total_tokens": 12640,
        "cost": 0.0009777,
        "is_byok": false,
        "prompt_tokens_details": {
   

## Extração de todos artigos

In [33]:
from pathlib import Path

md_dir = Path("Arquivos/md")

lista_titulo = []
lista_autores = []  
lista_ano_publicacao = []

print("Arquivos convertidos de pdf to md")

for arquivo_md in md_dir.glob("*.md"):
    print(f"\nLendo arquivo: {arquivo_md.name}")
    texto = arquivo_md.read_text(
    encoding="utf-8",
    errors="ignore"
    )
    print(type(texto))
    print(f"{len(texto)} caracteres")
    #print(texto[:5])   # primeiros 500 caracteres
    print("="*20)
    
    print("Conectando ao OpenRouter...")

    resposta_openrouter = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {openrouter_api_key}",
            "Content-Type": "application/json"
        },
        json={
            "model": "openai/gpt-4o-mini", # Este modelo nao suporta response_format ->"openai/gpt-3.5-turbo",
            "messages": [
                {
        "role": "system",
        "content": (
            "Extraia informações de artigos científicos."
        )
    },            

                {
                    "role": "user",
                    "content": texto,
                    
                }
            ],
            'response_format': {
                "type": "json_schema",
                "json_schema": {
                    "name": "article_info",
                    "schema": {
                        "type": "object",
                        "properties": {
                            "titulo": {"type": "string"},
                            "autores": {"type": "string"},
                            "ano_publicacao": {"type": "string"}
                        },
                        "required": ["titulo", "autores", "ano_publicacao"]
                    }
                }
            }
        },
        timeout=60
    )

    # Interrompe em erros HTTP, como 401, 404 ou 429
    try:
        resposta_openrouter.raise_for_status()
    except requests.exceptions.HTTPError:
        print("\nErro HTTP no OpenRouter:")
        print("Status:", resposta_openrouter.status_code)
        print("Resposta:", resposta_openrouter.text)
    else:
        resultado_openrouter = resposta_openrouter.json()

        if "choices" in resultado_openrouter:
            #print("\nResposta do OpenRouter")
            #print(
            #    resultado_openrouter["choices"][0]
            #    ["message"]["content"]
            #)
            data = json.loads(
                resultado_openrouter["choices"][0]["message"]["content"]
            )
            print("Dados extraídos do artigo")
            #display(pd.DataFrame([data]))
            lista_titulo.append(data.get("titulo", ""))
            lista_autores.append(data.get("autores", ""))
            lista_ano_publicacao.append(data.get("ano_publicacao", ""))
            
        elif "error" in resultado_openrouter:
            print("\nErro retornado pelo OpenRouter:")
            print(resultado_openrouter["error"])

        else:
            print("\nResposta inesperada do OpenRouter:")
            print(resultado_openrouter)

print("="*20)            
print('Dados extraídos com sucesso!')
data = {
    "titulo": lista_titulo,
    "autores": lista_autores,
    "ano_publicacao": lista_ano_publicacao
    }

display(pd.DataFrame(data))            

Arquivos convertidos de pdf to md

Lendo arquivo: bioetica_e_ia.md
<class 'str'>
51275 caracteres
Conectando ao OpenRouter...
Dados extraídos do artigo

Lendo arquivo: escrita_academica_ia.md
<class 'str'>
42740 caracteres
Conectando ao OpenRouter...
Dados extraídos do artigo

Lendo arquivo: twitter_algoritmo.md
<class 'str'>
54502 caracteres
Conectando ao OpenRouter...
Dados extraídos do artigo
Dados extraídos com sucesso!


,titulo,autores,ano_publicacao
0,Entre o algoritmo e o Juramento de Hipócrates:...,"Juracy Barbosa dos Santos, Guilhermina Rego, R...",2026
1,"Escrita acadêmica ética, responsável e humana ...",Rafael Cardoso Sampaio,2025
2,"O caso Twitter/X: algoritmo, espaço público e ...","Ettore Schimid Batalha, Jefferson Ribeiro da S...",2025
